# Experiment A: fallback gated detector diagnostic

Fast diagnostic for detector/fallback spam. OCR and QR are disabled deliberately.

Runtime: Google Colab GPU. Default guard requires one T4 GPU.
If Colab gives L4/A100 instead, change `EXP_REQUIRE_GPU_NAME`.

This notebook uses only local CV/OCR libraries inside Colab. It does not call cloud OCR/API/LLM during solution inference.

In [ ]:
!nvidia-smi
import sys
print(sys.version)

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

DATASET = "whitenigger/lenta-shelf-ai-bundle"
BUNDLE_DIR = Path("/content/lenta_bundle")
KAGGLE_INPUT = Path("/kaggle/input/lenta-shelf-ai-bundle")
KAGGLE_WORKING = Path("/kaggle/working")
RUN_TS = time.strftime("%Y%m%d_%H%M%S")
DRIVE_ROOT = Path("/content/drive/MyDrive/lenta_colab_runs")


def run(cmd, cwd=None, env=None, check=True):
    print("[RUN]", " ".join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, check=check)


def setup_drive():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        return True
    except Exception as exc:
        print("[WARN] Drive is not mounted:", repr(exc))
        return False


def setup_kaggle_credentials():
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / "kaggle.json"

    username = os.environ.get("KAGGLE_USERNAME")
    key = os.environ.get("KAGGLE_KEY")
    if not (username and key):
        try:
            from google.colab import userdata
            username = username or userdata.get("KAGGLE_USERNAME")
            key = key or userdata.get("KAGGLE_KEY")
        except Exception:
            pass
    if username and key:
        target.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
        target.chmod(0o600)
        print("[OK] Kaggle credentials loaded from env/Colab secrets")
        return

    candidates = list(Path("/content").glob("kaggle*.json"))
    if candidates:
        shutil.copy2(candidates[0], target)
        target.chmod(0o600)
        print("[OK] Kaggle credentials copied from", candidates[0])
        return

    from google.colab import files
    print("Upload kaggle.json now. The file stays inside this Colab runtime.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("kaggle.json was not uploaded")
    src = Path("/content") / next(iter(uploaded.keys()))
    shutil.copy2(src, target)
    target.chmod(0o600)
    print("[OK] Kaggle credentials uploaded")


def prepare_bundle():
    run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    setup_kaggle_credentials()
    BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
    if not (BUNDLE_DIR / "repo.zip").exists() or not (BUNDLE_DIR / "data.zip").exists():
        run(["kaggle", "datasets", "download", "-d", DATASET, "-p", str(BUNDLE_DIR), "--unzip"])
    if not (BUNDLE_DIR / "repo.zip").exists() or not (BUNDLE_DIR / "data.zip").exists():
        raise FileNotFoundError(f"Expected repo.zip and data.zip in {BUNDLE_DIR}")

    KAGGLE_INPUT.mkdir(parents=True, exist_ok=True)
    KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BUNDLE_DIR / "repo.zip", KAGGLE_INPUT / "repo.zip")
    shutil.copy2(BUNDLE_DIR / "data.zip", KAGGLE_INPUT / "data.zip")

    runner_root = Path("/content/lenta_runner_repo")
    if runner_root.exists():
        shutil.rmtree(runner_root)
    runner_root.mkdir(parents=True)
    with zipfile.ZipFile(BUNDLE_DIR / "repo.zip") as zf:
        zf.extractall(runner_root)
    script = runner_root / "repo" / "kaggle" / "gpu_experiment.py"
    if not script.exists():
        raise FileNotFoundError(script)
    print("[OK] runner:", script)
    return script


def run_experiment(script, exp_env):
    artifacts = KAGGLE_WORKING / "artifacts"
    if artifacts.exists():
        shutil.rmtree(artifacts)
    input_bundle = KAGGLE_WORKING / "input_bundle"
    if input_bundle.exists():
        shutil.rmtree(input_bundle)

    env = os.environ.copy()
    env.update({k: str(v) for k, v in exp_env.items()})
    run([sys.executable, str(script)], env=env)
    return artifacts


def run_error_analysis():
    work = Path("/tmp/lenta_shelf_ai_solution")
    eval_dir = work / "outputs" / "eval_public_fast"
    bundle = KAGGLE_WORKING / "input_bundle"
    if not eval_dir.exists() or not bundle.exists():
        print("[WARN] no eval outputs or input bundle for error analysis")
        return
    out_dir = KAGGLE_WORKING / "artifacts" / "error_analysis"
    out_dir.mkdir(parents=True, exist_ok=True)
    gt_by_stem = {p.stem: p for p in bundle.glob("data/**/*/*.csv")}
    for pred_csv in sorted(eval_dir.glob("*_recognized.csv")):
        stem = pred_csv.name.replace("_recognized.csv", "")
        gt_csv = gt_by_stem.get(stem)
        if gt_csv is None:
            print("[WARN] no GT csv for", pred_csv)
            continue
        run([
            sys.executable,
            "scripts/analyze_final_errors.py",
            "--gt-csv",
            str(gt_csv),
            "--pred-csv",
            str(pred_csv),
            "--out-json",
            str(out_dir / f"{stem}_error_analysis.json"),
        ], cwd=work)


def save_artifacts(artifacts, experiment_name):
    run_dir = DRIVE_ROOT / f"{experiment_name}_{RUN_TS}"
    if DRIVE_ROOT.exists():
        if run_dir.exists():
            shutil.rmtree(run_dir)
        shutil.copytree(artifacts, run_dir / "artifacts")
        archive = shutil.make_archive(str(run_dir), "zip", run_dir)
        print("[OK] saved to Drive:", run_dir)
        print("[OK] zip:", archive)
        return run_dir
    local_dir = Path("/content") / f"{experiment_name}_{RUN_TS}"
    if local_dir.exists():
        shutil.rmtree(local_dir)
    shutil.copytree(artifacts, local_dir / "artifacts")
    archive = shutil.make_archive(str(local_dir), "zip", local_dir)
    print("[OK] saved locally:", local_dir)
    print("[OK] zip:", archive)
    return local_dir


def print_key_reports(artifacts):
    paths = [
        artifacts / "run_summary.json",
        artifacts / "kaggle_metrics" / "detection_recall_yolo_only.json",
        artifacts / "kaggle_metrics" / "detection_recall_hybrid.json",
        artifacts / "eval_public_fast" / "metrics.json",
    ]
    for path in paths:
        if not path.exists():
            print("[MISS]", path)
            continue
        print("\n" + "=" * 90)
        print(path)
        print("=" * 90)
        print(path.read_text(encoding="utf-8")[:12000])

In [ ]:
EXP_ENV = {
    "EXP_NAME": "exp-colab-a-fallback-gated-noocr",
    "EXP_REQUIRE_GPU_NAME": "T4",
    "EXP_REQUIRE_GPU_COUNT": "1",
    "EXP_DEVICE": "0",
    "EXP_REQUIREMENTS": "requirements-train.txt",
    "EXP_SKIP_TORCH_PIN": "1",
    "EXP_SKIP_TRAIN": "1",
    "EXP_SKIP_PUBLIC_EVAL": "0",
    "EXP_RECALL_CONF": "0.08",
    "EXP_PIPELINE_ENABLE_OCR": "0",
    "EXP_PIPELINE_ENABLE_QR": "0",
    "EXP_PIPELINE_ENABLE_FALLBACKS": "1",
    "EXP_PIPELINE_YOLO_CONF": "0.08",
    "EXP_PIPELINE_SAMPLE_FPS": "4.0",
    "EXP_PIPELINE_MAX_DETECTIONS": "120",
    "EXP_PIPELINE_ENABLE_ZONAL_OCR": "0",
    "EXP_PIPELINE_FALLBACK_MIN_OBSERVATIONS": "3",
    "EXP_PIPELINE_FALLBACK_REQUIRE_EVIDENCE": "1",
    "EXP_PIPELINE_DEDUPE_EXTENDED_WINDOW_MS": "15000",
    "EXP_PIPELINE_DEDUPE_VISUAL_HASH": "12",
    "EXP_PIPELINE_DEDUPE_TEXT_SIMILARITY": "0.82"
}
                EXPERIMENT_NAME = EXP_ENV["EXP_NAME"]

                setup_drive()
                script = prepare_bundle()
                artifacts = run_experiment(script, EXP_ENV)
                run_error_analysis()
                run_dir = save_artifacts(artifacts, EXPERIMENT_NAME)
                print_key_reports(artifacts)

## Required critique after run

If rows explode again, strict fallback gating is still insufficient. If rows collapse to zero, fallback evidence gate is too strict for no-OCR mode.

Minimum evidence to call this useful:
- `eval_public_fast/metrics.json` has nonzero `good_rows_at_80`;
- `26_12-20 pred_rows` moves closer to GT without collapsing `matched_rows`;
- `field_fill` in error-analysis improves for prices, barcode, QR, SKU;
- no fallback spam returns.